In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!nvidia-smi

Thu Jun 11 08:38:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install transformers torchaudio "onnxruntime==1.20.1" "onnx==1.20.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 100.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 87.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 102.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.1 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
  Attempting uninstall: onnx
    Found existing installation: onnx 1.21.0
    Uninstalling onnx-1.21.0:
      Successfully uninstalled onnx-1.21.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.

In [3]:
from huggingface_hub import login

hf_token = "Your_token_here"

login(token=hf_token)
print("Successfully logged into Hugging Face Hub!")

Successfully logged into Hugging Face Hub!


In [4]:
from transformers import AutoModel
import torch
import torchaudio

# 1. Load the model
print("Loading the model... (This will take a few minutes on first run)")
model = AutoModel.from_pretrained("ai4bharat/indic-conformer-600m-multilingual", trust_remote_code=True)

# Move model to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Model loaded on {device}")



Loading the model... (This will take a few minutes on first run)


config.json:   0%|          | 0.00/241 [00:00<?, ?B/s]

model_onnx.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indic-conformer-600m-multilingual:
- model_onnx.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Please check FRAME_DURATION_MS. The timestamps can be inaccurate
Please check FRAME_DURATION_MS. The timestamps can be inaccurate


Fetching 404 files:   0%|          | 0/404 [00:00<?, ?it/s]

Please check FRAME_DURATION_MS. The timestamps can be inaccurate


/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:115: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Model loaded on cuda


In [5]:
!pip install -q datasets huggingface_hub

from datasets import load_dataset

telugu_valid = load_dataset(
    "ai4bharat/IndicVoices",
    "telugu",
    split="valid",        # the split name inside the config; usually “train” for the data split
    streaming=True       # streams rows on‑demand
)


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/88 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/56 [00:00<?, ?it/s]

In [6]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 36.0 MB/s eta 0:00:0000:0100:01


In [10]:
import json
import os
from jiwer import process_words # Only import process_words
from tqdm import tqdm
import torch
import torchaudio
import numpy as np

# ── Checkpoint settings ──────────────────────────────────────────────────────
CHECKPOINT_PATH = "/kaggle/working/checkpoint_telugu.json"
CHECKPOINT_EVERY = 100          # save every N samples

def save_checkpoint(idx, refs, preds_ctc, preds_rnnt, path=CHECKPOINT_PATH):
    data = {
        "last_index": idx,
        "references": refs,
        "predictions_ctc": preds_ctc,
        "predictions_rnnt": preds_rnnt,
    }
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)          # atomic write – avoids partial files
    print(f"  ✔ Checkpoint saved at sample {idx} → {path}")

def load_checkpoint(path=CHECKPOINT_PATH):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"Resuming from checkpoint: {data['last_index'] + 1} samples already done.")
        return data["last_index"] + 1, data["references"], data["predictions_ctc"], data["predictions_rnnt"]
    return 0, [], [], []
# ─────────────────────────────────────────────────────────────────────────────

def load_audio2(audio_source, target_sr=16000):
    if isinstance(audio_source, tuple) and len(audio_source) == 2:
        waveform_np, original_sr = audio_source
        waveform = torch.from_numpy(waveform_np).float().unsqueeze(0)
    else:
        waveform, original_sr = torchaudio.load(audio_source)

    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    if original_sr != target_sr:
        resampler = torchaudio.transforms.Resample(orig_freq=original_sr, new_freq=target_sr)
        waveform = resampler(waveform)
    return waveform

# Load checkpoint (starts from 0 if none exists)
start_idx, references, predictions_ctc, predictions_rnnt = load_checkpoint()

print(f"Starting inference and metric collection for set1... (from index {start_idx})")

# Skip already-processed samples
current_set1_iterable = telugu_valid.take(3295)
if start_idx > 0:
    import itertools
    current_set1_iterable = itertools.islice(current_set1_iterable, start_idx, None)
    print(f"Skipped first {start_idx} samples.")

language_code = "te"

for sample_idx, sample in tqdm(enumerate(current_set1_iterable, start=start_idx), initial=start_idx, total=3295):
    audio_data_decoded = sample["audio_filepath"]
    reference_text = sample["text"]

    try:
        audio_input = load_audio2((audio_data_decoded['array'], audio_data_decoded['sampling_rate'])).to(device)

        with torch.no_grad():
            transcription_ctc  = model(audio_input, language_code, "ctc")
            transcription_rnnt = model(audio_input, language_code, "rnnt")

        references.append(reference_text)
        predictions_ctc.append(transcription_ctc.strip())
        predictions_rnnt.append(transcription_rnnt.strip())

    except RuntimeError as e:
        print(f"Skipping sample {sample_idx} due to audio processing error: {e}.")
        continue

    # ── Save checkpoint every CHECKPOINT_EVERY samples ───────────────────────
    if (sample_idx + 1) % CHECKPOINT_EVERY == 0:
        save_checkpoint(sample_idx, references, predictions_ctc, predictions_rnnt)
    # ─────────────────────────────────────────────────────────────────────────

# Final checkpoint after the loop completes
save_checkpoint(3294, references, predictions_ctc, predictions_rnnt)
print("Inference for set1 complete. Calculating metrics...")


Resuming from checkpoint: 200 samples already done.
Starting inference and metric collection for set1... (from index 200)
Skipped first 200 samples.


  9%|▉         | 300/3295 [01:40<46:00,  1.08it/s]  

  ✔ Checkpoint saved at sample 299 → /kaggle/working/checkpoint_telugu.json


 12%|█▏        | 400/3295 [03:43<35:47,  1.35it/s]  

  ✔ Checkpoint saved at sample 399 → /kaggle/working/checkpoint_telugu.json


...

  ✔ Checkpoint saved at sample 3099 → /kaggle/working/checkpoint_telugu.json


 97%|█████████▋| 3200/3295 [1:27:26<04:34,  2.89s/it]

  ✔ Checkpoint saved at sample 3199 → /kaggle/working/checkpoint_telugu.json


100%|██████████| 3295/3295 [1:31:36<00:00,  1.78s/it]

  ✔ Checkpoint saved at sample 3294 → /kaggle/working/checkpoint_telugu.json
Inference for set1 complete. Calculating metrics...


In [12]:
from jiwer import wer, cer

# ── CTC metrics ──────────────────────────────
ctc_wer = wer(references, predictions_ctc)
ctc_cer = cer(references, predictions_ctc)

# ── RNN-T metrics ────────────────────────────
rnnt_wer = wer(references, predictions_rnnt)
rnnt_cer = cer(references, predictions_rnnt)

# ── print results ────────────────────────────
print(f"{'Metric':<10} {'CTC':>10} {'RNN-T':>10}")
print("-" * 32)
print(f"{'WER':<10} {ctc_wer*100:>9.2f}% {rnnt_wer*100:>9.2f}%")
print(f"{'CER':<10} {ctc_cer*100:>9.2f}% {rnnt_cer*100:>9.2f}%")

Metric            CTC      RNN-T
--------------------------------
WER            27.86%     26.28%
CER             8.53%      8.42%
